In [25]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal

from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate


In [3]:
load_dotenv()

True

In [2]:
model = ChatGroq(
    model_name="qwen/qwen3.8-27b"
)


In [4]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal

In [19]:
class LLMState(TypedDict):
    topic:str
    report:str
    blog:str

In [20]:
def LLM1(state:LLMState)->LLMState:
     
     topic=state['topic']

     prompt= PromptTemplate(
        template=f'Give me a detailed report on the following topic: {topic}',
        input_variables=['topic']

     )

     chain= prompt | model 

     report= chain.invoke({'topic':topic}).content

     state['report']= report

     return state





In [31]:
def LLM2(state: LLMState)->LLMState:

    report= state['report']

    prompt= PromptTemplate(
         template= f'Write a 300 words blog based on following report: {report}',
         input_variables=['report']

    )

    chain= prompt | model

    response= chain.invoke({'report':report}).content
    
    state['blog']= response.content

    return {"blog": response.content}

    





In [32]:
graph= StateGraph(LLMState)

graph.add_node('LLM1',LLM1)
graph.add_node('LLM2',LLM2)

graph.add_edge(START,'LLM1')
graph.add_edge('LLM1', 'LLM2')
graph.add_edge('LLM2', END)


workflow= graph.compile()


In [35]:
initial_state={'topic':'AI Evolution'}


response_state = workflow.invoke(initial_state)

print(response_state)



RateLimitError: Error code: 429 - {'error': {'message': "Request too large for model `qwen/qwen3.8-27b` in organization `org_01kpb63cs5f87acasqyah4y416` service tier `on_demand` on output tokens per minute (OTPM): Limit 1000, Requested 2048. The request's expected output tokens exceed the enforced limit; reduce max_tokens (or the request's expected output) and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing", 'type': 'tokens', 'code': 'rate_limit_exceeded'}}